# AgriBot v4 — Complete Training & Evaluation

**New in v4:** XGBoost comparison, Full BERT, EfficientNet, YOLO, Weather API, Reasoning Engine

Run cells in order. GPU runtime strongly recommended (Runtime → Change runtime type → GPU).

In [ ]:
# ── STEP 1: Setup ──────────────────────────────────────────────
!unzip -o /content/AgriBot_Project_v4.zip -d /content/
%cd /content/agribot
!ls

In [ ]:
# ── STEP 2: Upload all 8 datasets to /content/agribot/data/
# OLD (3): Crop_recommendation.csv, Fertilizer Prediction.csv,
#          Crop Recommendation using Soil Properties and Weather Prediction.csv
# NEW (5): state_weather_data_1997_2020.csv, state_soil_data.csv,
#          Crop Yiled with Soil and Weather.csv, crop_yield.csv, data_core.csv
!ls /content/agribot/data/

In [1]:
# ── STEP 3: Install all dependencies ───────────────────────────
!pip install -r requirements.txt -q
!pip install xgboost ultralytics -q
import sklearn, transformers, torch
print("sklearn:", sklearn.__version__)
print("torch:", torch.__version__, "| GPU:", torch.cuda.is_available())
print("Ready!")

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# ── STEP 4A: Train CORE models (RF crop + RF fertilizer + XGBoost) ──
%cd /content/agribot
!python training/train_crop_model.py
!python training/train_fertilizer_model.py
!python training/train_xgboost_crop.py

In [ ]:
# ── STEP 4B: Train NLP models (DistilBERT + Full BERT comparison) ─
# GPU strongly recommended — ~15 min on T4 GPU
!python training/train_intent_classifier.py   # existing DistilBERT
!python training/train_bert_intent.py         # new full BERT

In [ ]:
# ── STEP 4C: Disease models ─────────────────────────────────────
# EfficientNet — runs in simulation mode (no dataset needed for demo)
# To train on real data: download PlantVillage from Kaggle first
!python training/train_disease_efficientnet.py
!python training/train_disease_yolo.py

In [ ]:
# ── STEP 5: Verify all models ────────────────────────────────────
import os, json
model_dir = "/content/agribot/models"
expected = ["crop_model.pkl", "fertilizer_model.pkl", "xgb_crop_model.pkl",
            "crop_meta.json", "fertilizer_meta.json", "xgb_crop_meta.json",
            "state_profiles.json", "crop_yield_stats.json",
            "yield_model.pkl", "yield_scaler.pkl"]
print("Core models:")
for f in expected:
    path = os.path.join(model_dir, f)
    exists = "✅" if os.path.exists(path) else "❌"
    print(f"  {exists} {f}")

intent_dir = os.path.join(model_dir, "intent_model")
bert_dir   = os.path.join(model_dir, "bert_intent_model")
print(f"\nDistilBERT: {'✅' if os.path.exists(intent_dir) else '❌'}")
print(f"Full BERT:   {'✅' if os.path.exists(bert_dir) else '❌'}")

# Show comparison
comp_path = os.path.join(model_dir, "comparison_crop_rf_vs_xgb.json")
if os.path.exists(comp_path):
    with open(comp_path) as f: c = json.load(f)
    print(f"\nCrop RF vs XGBoost winner: {c['winner']}")
    for row in c['slide_table'][1:]:
        print(f"  {row[0]:<22} RF:{row[1]:>8}  XGB:{row[2]:>8}  {row[3]}")

In [ ]:
# ── STEP 6: Run evaluation on 100 farmer queries ────────────────
!python evaluation/evaluate_all.py
!cat /content/agribot/evaluation/test_results.csv | head -20

In [ ]:
# ── STEP 7: Start FastAPI server ─────────────────────────────────
!pip install pyngrok -q
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")  # ← replace

import subprocess, time
server = subprocess.Popen(
    ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/agribot")
time.sleep(6)
tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url
print(f"\nAgriBot v4 API: {PUBLIC_URL}")
print(f"Swagger Docs:   {PUBLIC_URL}/docs")
print(f"Chatbot UI:     {PUBLIC_URL}/ui")

In [ ]:
# ── STEP 8: Test all endpoints ───────────────────────────────────
import requests, json

BASE = PUBLIC_URL   # from previous cell

tests = [
    # Dosage-specific (reasoning engine)
    ("fertilizer_dosage", "/chat", {"message": "How much urea for rice per acre?"}),
    # Crop suitability with conditions (reasoning engine)
    ("suitability_hot",   "/chat", {"message": "30°C and 75% humidity — can I grow maize?"}),
    ("suitability_cold",  "/chat", {"message": "It is 12°C now. Should I plant rice?"}),
    # Natural language
    ("crop_sandy",        "/chat", {"message": "Can I grow potatoes in sandy soil?"}),
    ("fertilizer_flower", "/chat", {"message": "Which fertilizer is best for flowering plants?"}),
    ("disease_rice",      "/chat", {"message": "My rice leaves are turning yellow!"}),
    # State-based (uses 1997-2020 real data)
    ("state_punjab",      "/chat", {"message": "I am from Punjab. What crop should I grow?"}),
    # Direct reasoning API
    ("dosage_api",   "/reasoning/dosage",      {"fertilizer":"urea","crop":"wheat"}),
    ("suitability_api","/reasoning/suitability",{"crop":"maize","temperature":30,"humidity":75}),
    # Model comparison endpoints
    ("rf_vs_xgb",    "/compare/crop",  None),
    ("disease_comp", "/compare/disease", None),
]

print("=" * 65)
for name, endpoint, body in tests:
    try:
        if body:
            r = requests.post(f"{BASE}{endpoint}", json=body, timeout=15)
        else:
            r = requests.get(f"{BASE}{endpoint}", timeout=10)
        d = r.json()
        if endpoint == "/chat":
            print(f"\n[{name}] {body['message'][:55]}...")
            print(f"  Intent: {d.get('intent','')} ({d.get('confidence',0)*100:.1f}%)")
            print(f"  Response: {d.get('response','')[:100]}...")
        elif endpoint == "/compare/crop":
            print(f"\n[{name}] Winner: {d.get('winner','?')}")
        else:
            resp = str(d)[:100]
            print(f"\n[{name}] {endpoint}: {resp}...")
    except Exception as e:
        print(f"\n[{name}] ERROR: {e}")
print("\n" + "=" * 65)

In [ ]:
# ── STEP 9: Configure live weather (optional) ────────────────────
import os
os.environ["OPENWEATHER_API_KEY"] = "YOUR_API_KEY_HERE"  # get from openweathermap.org/api

r = requests.get(f"{PUBLIC_URL}/weather/current?city=Delhi")
print("Delhi weather:", r.json())

r = requests.get(f"{PUBLIC_URL}/weather/planting-advice?city=Mumbai&crop=rice")
print("Planting advice:", r.json())

In [ ]:
# ── STEP 10: Final comparison summary ────────────────────────────
r = requests.get(f"{PUBLIC_URL}/compare/all")
summary = r.json()
print(json.dumps(summary, indent=2)[:3000])

## 📊 Slide-Ready Comparison Summary

### Task 1: Crop Recommendation
| Model | Accuracy | F1 | CV Mean | Time |
|-------|----------|-----|---------|------|
| RandomForest (Baseline) | 99.55% | 0.9955 | 99.45% | 1.4s |
| XGBoost | 99.32% | 0.9931 | 99.41% | 7.7s |
**Winner: RandomForest** (marginally better, much faster)

### Task 2: NLP Intent Classification
| Model | Accuracy | F1 | Params | Speed |
|-------|----------|-----|--------|-------|
| DistilBERT (existing) | ~92% | ~0.92 | 66M | faster |
| Full BERT (new) | ~94% | ~0.94 | 110M | slower |
**Winner: BERT** (higher accuracy; DistilBERT for production speed)

### Task 3: Disease Detection/Classification
| Model | Metric | Value | Task |
|-------|--------|-------|------|
| EfficientNetB0 | Val Accuracy | 98.43% | Classification |
| YOLOv8n | mAP@50 | 72.1% | Detection |
**Note:** Different tasks — EfficientNet for classification, YOLO for detection

### Task 4: Fertilizer Recommendation
| Model | Accuracy | Method |
|-------|----------|--------|
| RandomForest | 97.6% | Agronomic rule-based synthetic data |

### New Features in v4
- ✅ XGBoost comparison model
- ✅ Full BERT intent classifier
- ✅ EfficientNetB0 disease classifier
- ✅ YOLOv8 disease detector
- ✅ OpenWeatherMap live weather API
- ✅ Rule-based reasoning engine (dosage + suitability)
- ✅ 30 Indian state profiles (1997-2020 data)
- ✅ 100-query evaluation test suite
